In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4

In [ ]:
!pip install faiss-cpu langchain-huggingface pypdf sentence-transformers torch

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import re

In [ ]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [ ]:
# read the pdf 

pdf_path="/kaggle/input/datasets/mohammedosman123/paser-test/sample_cv_sarah_johnson.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [ ]:
name_schema = ResponseSchema(
    name="full_name",
    description="the full name of the person in the cv"
)
email_schema = ResponseSchema(
    name="email",
    description="The email of the person in the cv"
)
education_schema = ResponseSchema(
    name="education",
    description="the list of education and certificates the person in the cv has recieved"
)
skills_schema = ResponseSchema(
    name="skills",
    description="the list of skills a person has recieved"
)
experience_schema = ResponseSchema(
    name="experience",
    description="the list of roles in a company and how much years "
)

response_schemas = [name_schema, email_schema,education_schema,skills_schema,experience_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

cv_extraction_template = """
You are a smart assistant that extracts:
full_name (string)
email (string)
education → list of {{degree, institution, year}}
skills → list of strings
experience → list of {{role, company, years}}

Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
John Smith – john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)

Now extract from the following input:
"{user_input}"
"""

In [ ]:
user_input =documents

prompt = PromptTemplate(
    template=cv_extraction_template,
    input_variables=["user_input", "format_instructions"]
    ).format(user_input=user_input, format_instructions=format_instructions)

response = generate_text(prompt, max_length=5000)[0]
print(response)

In [ ]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [ ]:
json_text = extract_json_block(response)
print(json_text)

In [ ]:
!pip install -q fastapi uvicorn pyngrok python-multipart

In [ ]:
import re, json, tempfile, threading, traceback, socket
import uvicorn
from fastapi import FastAPI, UploadFile, File
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# find a free port automatically
s = socket.socket()
s.bind(("", 0))
PORT = s.getsockname()[1]
s.close()


def parse_cv(pdf_path):
    documents = PyPDFLoader(pdf_path).load()
    cv_text = "\n".join(d.page_content for d in documents)

    prompt = PromptTemplate(
        template=cv_extraction_template,
        input_variables=["user_input", "format_instructions"],
    ).format(user_input=cv_text, format_instructions=format_instructions)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    text = match.group(1) if match else response
    try:
        return json.loads(text)
    except Exception:
        return {"error": "Could not parse output", "raw": response}


api_new = FastAPI()

@api_new.post("/parse")
async def parse(file: UploadFile = File(...)):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as f:
            f.write(await file.read())
        return parse_cv(f.name)
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return {"error": f"{type(e).__name__}: {e}", "raw": tb}


threading.Thread(
    target=lambda: uvicorn.run(api_new, host="0.0.0.0", port=PORT), daemon=True
).start()

ngrok.kill()
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
print("port:", PORT)
print(ngrok.connect(PORT).public_url)